In [19]:
import os
import sys
import torch
import subprocess
from glob import glob

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from osgeo import gdal

%matplotlib inline

In [20]:
!pip install --upgrade transformers
from transformers import AutoImageProcessor

Defaulting to user installation because normal site-packages is not writeable


In [21]:
repo_dir = "lfm"

sys.path.append("lfm")

from lfm.tasks.inst_segmentation_m2f.dataset import get_dataloaders, calculate_dataset_statistics
from lfm.tasks.inst_segmentation_m2f.driver import train_model, prepare_image_for_display, convert_binary_masks_to_instance_map
from lfm.tasks.inst_segmentation_m2f.mask2former_model import create_mask2former_dinov3_model

In [22]:
# Data paths
INPUT_DIR = "/explore/nobackup/projects/lfm/inst_seg_viz_chips_3_band"
IMAGE_DIR = f""
LABEL_DIR = f""

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs_meeting"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs_meeting
Using device: cuda


In [23]:
print("\n" + "="*60)
print("STEP 1: Computing dataset statistics.")
print("="*60)
# MEAN, STD = calculate_dataset_statistics(IMAGE_DIR)

# # Check if statistics already exist
# mean_path = os.path.join(OUTPUT_DIR, "dataset_mean.npy")
# std_path = os.path.join(OUTPUT_DIR, "dataset_std.npy")

# # Save statistics
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# np.save(mean_path, MEAN)
# np.save(std_path, STD)
# print(f"✓ Saved statistics to {OUTPUT_DIR}")
MEAN = np.load("/explore/nobackup/people/ajkerr1/Lunar_FM/model_view_cont/lfm/notebooks/outputs/dataset_mean.npy")
STD = np.load("/explore/nobackup/people/ajkerr1/Lunar_FM/model_view_cont/lfm/notebooks/outputs/dataset_std.npy")
print(MEAN, STD)


STEP 1: Computing dataset statistics.
[0.32744208 0.32249309 0.302985  ] [0.15045737 0.15014801 0.14386101]


In [24]:
# Determine base model name (adjust if using different size)
print("\nSTEP 2: Creating image processor.")
print("="*60)
BASE_MODEL = "facebook/mask2former-swin-large-coco-instance"

print("\nCreating image processor with custom normalization...")
print(f"  Base model: {BASE_MODEL}")
print(f"  Target size: {TARGET_SIZE}")
print(f"  Custom mean: {MEAN.tolist()}")
print(f"  Custom std: {STD.tolist()}")

# Create image processor with your dataset's statistics
image_processor = AutoImageProcessor.from_pretrained(
    BASE_MODEL,
    do_resize=True,
    size={"height": TARGET_SIZE[0], "width": TARGET_SIZE[1]},
    # do_normalize=True,
    # image_mean=MEAN.tolist(),  # Use YOUR dataset statistics
    # image_std=STD.tolist(),     # Use YOUR dataset statistics
    do_reduce_labels=False,     # Keep background as 0
)


STEP 2: Creating image processor.

Creating image processor with custom normalization...
  Base model: facebook/mask2former-swin-large-coco-instance
  Target size: (304, 304)
  Custom mean: [0.32744208232738664, 0.32249308820269185, 0.3029849963897787]
  Custom std: [0.1504573733079846, 0.1501480130785911, 0.14386101048440858]


In [25]:
# Required for mask2former model
label2id = {
    "background": 0,
    "crater": 1,
}
id2label = {v: k for k, v in label2id.items()}

In [26]:
def load_model_with_new_classes(checkpoint_path: str, model_config, 
                                device='cuda', hub_token=None, verbose=True):
    """
    Load model weights from checkpoint, skipping classification head layers
    that don't match (useful when changing number of classes).
    
    Args:
        checkpoint_path: Path to checkpoint
        model_config: Config dict with YOUR desired number of classes
        device: Device to load on
        hub_token: HuggingFace token
        verbose: Print detailed loading information
    
    Returns:
        model: Model with backbone weights loaded, new classification head
        metadata: Dictionary with loading info and training history
    """
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    checkpoint_state = checkpoint['model_state_dict']
    
    if verbose:
        # Detect checkpoint's number of classes
        ckpt_num_classes = checkpoint_state['class_predictor.weight'].shape[0]
        print(f"\n{'='*70}")
        print(f"LOADING CHECKPOINT WITH PARTIAL WEIGHTS")
        print(f"{'='*70}")
        print(f"Checkpoint classes: {ckpt_num_classes}")
        print(f"Target classes: {len(model_config['label2id'])}")
        print(f"Checkpoint epoch: {checkpoint['epoch']}")
    
    # Create NEW model with YOUR desired number of classes
    model = create_mask2former_dinov3_model(
        label2id=model_config['label2id'],
        id2label=model_config['id2label'],
        expected_channels=model_config.get('expected_channels', [96, 192, 384, 768]),
        freeze_backbone=False,
        hub_token=hub_token,
    )
    
    model_state = model.state_dict()
    
    # Keys to skip (classification head components)
    skip_patterns = [
        'class_predictor',  # Main classification layer
        'criterion.empty_weight',  # Loss criterion for empty class
    ]
    
    # Filter weights
    loaded_keys = []
    skipped_keys = []
    shape_mismatches = []
    
    for key, value in checkpoint_state.items():
        # Check if this key should be skipped
        should_skip = any(pattern in key for pattern in skip_patterns)
        
        if should_skip:
            skipped_keys.append(key)
            if verbose:
                if key in model_state:
                    print(f"⚠️  Skipping {key:60s} [shape: {list(value.shape)} -> {list(model_state[key].shape)}]")
                else:
                    print(f"⚠️  Skipping {key:60s} [not in new model]")
        elif key in model_state:
            if model_state[key].shape == value.shape:
                loaded_keys.append(key)
            else:
                shape_mismatches.append(key)
                skipped_keys.append(key)
                if verbose:
                    print(f"❌ Shape mismatch {key:60s} [ckpt: {list(value.shape)}, model: {list(model_state[key].shape)}]")
        else:
            skipped_keys.append(key)
            if verbose:
                print(f"❌ Key not in model: {key}")
    
    # Load matching weights
    filtered_state = {k: v for k, v in checkpoint_state.items() if k in loaded_keys}
    missing_keys, unexpected_keys = model.load_state_dict(filtered_state, strict=False)
    
    model.to(device)
    
    # Summary
    if verbose:
        print(f"\n{'='*70}")
        print(f"LOADING SUMMARY")
        print(f"{'='*70}")
        print(f"✅ Loaded layers:     {len(loaded_keys):4d} / {len(checkpoint_state)}")
        print(f"⚠️  Skipped layers:    {len(skipped_keys):4d} / {len(checkpoint_state)}")
        print(f"📊 Loading percentage: {100 * len(loaded_keys) / len(checkpoint_state):.1f}%")
        print(f"\n⚠️  WARNING: Classification head uses RANDOM initialization!")
        print(f"   You should fine-tune the model on your dataset.")
        print(f"{'='*70}\n")
    
    # Prepare metadata
    metadata = {
        'checkpoint_epoch': checkpoint['epoch'],
        'train_losses': checkpoint['train_losses'],
        'val_losses': checkpoint['val_losses'],
        'loaded_keys': loaded_keys,
        'skipped_keys': skipped_keys,
        'shape_mismatches': shape_mismatches,
        'loading_percentage': 100 * len(loaded_keys) / len(checkpoint_state),
    }
    
    return model, metadata


# Usage with 2 classes:
model_config = {
    'label2id': {'background': 0, 'foreground': 1},  # Your 2 classes
    'id2label': {0: 'background', 1: 'foreground'},
    'expected_channels': [96, 192, 384, 768],
}

ckpt_path = "/explore/nobackup/people/ajkerr1/Lunar_FM/model_view_cont/lfm/notebooks/outputs/checkpoints/checkpoint_epoch_050.pt"
model, metadata = load_model_with_new_classes(
    ckpt_path,
    model_config,
    device='cuda',
    verbose=True
)

print(f"Model loaded! {metadata['loading_percentage']:.1f}% of weights transferred.")

/explore/nobackup/people/ajkerr1/.nccstmp/ipykernel_1944304/3428143148.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_loca


LOADING CHECKPOINT WITH PARTIAL WEIGHTS
Checkpoint classes: 3
Target classes: 2
Checkpoint epoch: 50


Loading weights:   0%|          | 0/782 [00:00<?, ?it/s]

Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-large-coco-instance
Key                    | Status   |                                                                                        
-----------------------+----------+----------------------------------------------------------------------------------------
class_predictor.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81, 256]) vs model:torch.Size([3, 256])
class_predictor.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81]) vs model:torch.Size([3])          
criterion.empty_weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([81]) vs model:torch.Size([3])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

⚠️  Skipping class_predictor.weight                                       [shape: [3, 256] -> [3, 256]]
⚠️  Skipping class_predictor.bias                                         [shape: [3] -> [3]]
⚠️  Skipping criterion.empty_weight                                       [shape: [3] -> [3]]

LOADING SUMMARY
✅ Loaded layers:     1202 / 1205
⚠️  Skipped layers:       3 / 1205
📊 Loading percentage: 99.8%

⚠️  WARNING: Classification head uses RANDOM initialization!
   You should fine-tune the model on your dataset.

Model loaded! 99.8% of weights transferred.


In [27]:
# import numpy as np
# from osgeo import gdal
# from glob import glob

# tiff_dir = f"/explore/nobackup/people/rlgill/SystemTesting/lunar/newCubes/"
# tiffs = glob(f"{tiff_dir}/*.tif")
# print(len(tiffs))
# images = []
# target_size = 304

# for tiff in tiffs:
#     dataset = gdal.Open(tiff)
#     num_bands = dataset.RasterCount
#     print(f"Loaded tiff with bands: {num_bands}")
    
#     # Read first 3 bands directly (bands are 1-indexed in GDAL)
#     # Returns shape: (3, height, width)
#     img_array = dataset.ReadAsArray(band_list=[1, 2, 3])
    
#     # Center crop to (3, 304, 304)
#     h, w = img_array.shape[1], img_array.shape[2]  # 512, 512
#     start_h = (h - target_size) // 2  # 104
#     start_w = (w - target_size) // 2  # 104
#     img_cropped = img_array[:, start_h:start_h+target_size, start_w:start_w+target_size]
    
#     # Transpose to (height, width, channels) format: (304, 304, 3)
#     img_cropped = np.transpose(img_cropped, (1, 2, 0))
#     images.append(img_cropped)
    
#     dataset = None  # Close dataset

# images = np.array(images)  # Shape: (N, 304, 304, 3)

# print(images.shape)

# Convert to torch tensor if needed (PyTorch expects channels-first)
# images_tensor = torch.from_numpy(images).permute(0, 3, 1, 2).float()  # Shape: (N, 3, 304, 304)

# Now use the model
# with torch.no_grad():
#     outputs = model(images_tensor)

In [58]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from osgeo import gdal
from glob import glob
import os
import xarray as xr

def extract_images(tiff_paths, num_images_to_extract, bands_per_image=7):
    images_per_tiff = []
    for path in tiff_paths:
        with xr.open_dataset(path, engine='rasterio') as ds:
            images_per_tiff.append(ds.sizes['band'] // bands_per_image)
            # print(f"Shape of image: {ds.dims}")

    # print("images to extract per tiff:", images_per_tiff)
    cumulative = np.cumsum([0] + images_per_tiff)
    # print("cumsum:", cumulative)
    
    if num_images_to_extract > cumulative[-1]:
        raise ValueError(f"Requested {num_images_to_extract} images but only {cumulative[-1]} available")
    
    extracted = []
    for img_idx in range(num_images_to_extract):
        tiff_idx = np.searchsorted(cumulative[1:], img_idx, side='right')
        img_in_tiff = img_idx - cumulative[tiff_idx]
        start_band = img_in_tiff * bands_per_image
        
        with xr.open_dataset(tiff_paths[tiff_idx], engine='rasterio') as ds:
            sliced = ds.isel(band=slice(start_band, start_band + bands_per_image))
            data_var = list(sliced.data_vars)[0]
            extracted.append(sliced[data_var].values)

    for elem in extracted[:5]:
        print(f"Extracted item shape: {elem.shape}")
    return np.array(extracted)
    

def create_binary_colormap(instance_mask):
    """
    Create a simple binary colormap: 0 -> black, any other value -> red.
    
    Args:
        instance_mask: (H, W) array with instance IDs (0 = background)
        
    Returns:
        colored: (H, W, 3) RGB array with values in [0, 1]
    """
    h, w = instance_mask.shape
    colored = np.zeros((h, w, 3), dtype=np.float32)
    
    # Set all non-zero pixels to red
    mask = instance_mask != 0
    colored[mask] = [1.0, 0.0, 0.0]  # Pure red
    
    return colored


def visualize_inference_from_tiffs(
    model,
    image_processor,
    device,
    tiff_dir,
    mean,
    std,
    output_path="inference_test.png",
    n_images=20,
    target_size=304,
    threshold=0.5,
    normalize=True,
):
    """
    Load TIFF images, run inference, and visualize results.
    
    Args:
        model: Trained model
        image_processor: Image processor for post-processing
        device: torch device
        tiff_dir: Directory containing TIFF files
        mean: Dataset mean per channel (from training stats)
        std: Dataset std per channel (from training stats)
        output_path: Path to save visualization
        n_images: Number of images to visualize
        target_size: Size to crop/resize images to
        threshold: Confidence threshold for predictions
    """
    model.eval()
    
    # Load TIFF files
    tiffs = glob(f"{tiff_dir}/*.tif")
    print(f"Found {len(tiffs)} TIFF files")
    
    if len(tiffs) == 0:
        raise ValueError(f"No TIFF files found in {tiff_dir}")
    
    # Limit to n_samples
    # tiffs = tiffs[:n_samples]
    
    images = []
    filenames = []

    # Load numpy array of 7-band images
    print(f"Loading {n_images} images from TIFF files...")
    images_npy = extract_images(tiffs, n_images, bands_per_image=7)
    print(images_npy.shape)
    return
    print(f"Loaded images shape: {images.shape}")

    images_reshaped = 
    
    # Normalize images using training statistics
    mean_reshaped = mean.reshape(1, 1, 1, 3)  # (1, 1, 1, 3)
    std_reshaped = std.reshape(1, 1, 1, 3)    # (1, 1, 1, 3)
    images_normalized = (images - mean_reshaped) / std_reshaped if normalize else images
    
    # Convert to torch tensor (channels-first: N, C, H, W)
    images_tensor = torch.from_numpy(images_normalized).permute(0, 3, 1, 2).float()
    images_tensor = images_tensor.to(device)
    
    print(f"Running inference on {len(images_tensor)} images...")
    
    # Run inference
    preds_list = []
    with torch.no_grad():
        batch_size = images_tensor.shape[0]
        target_sizes = [(target_size, target_size)] * batch_size
        
        outputs = model(pixel_values=images_tensor)
        
        # Post-process
        try:
            post_processed = image_processor.post_process_instance_segmentation(
                outputs,
                threshold=threshold,
                target_sizes=target_sizes,
                return_binary_maps=True,
            )
        except Exception as e:
            print(f"Warning: Post-processing failed: {e}")
            post_processed = image_processor.post_process_instance_segmentation(
                outputs,
                threshold=threshold,
                target_sizes=target_sizes,
            )
        
        # Convert to instance masks
        for idx, result in enumerate(post_processed):
            if "segmentation" in result:
                instance_mask = result["segmentation"]
                if isinstance(instance_mask, torch.Tensor):
                    instance_mask = instance_mask.cpu().numpy()
                instance_mask = convert_binary_masks_to_instance_map(instance_mask)
                preds_list.append(instance_mask)
            else:
                print(f"Warning: 'segmentation' not in result for image {idx}")
                instance_mask = np.zeros((target_size, target_size), dtype=np.int32)
                preds_list.append(instance_mask)
    
    # Create visualization
    print("Creating visualization...")
    
    batch_size = len(images)
    fig, axes = plt.subplots(2, batch_size, figsize=(5 * batch_size, 10))
    
    if batch_size == 1:
        axes = axes.reshape(-1, 1)
    
    for i in range(batch_size):
        img = images[i]  # Original unnormalized image (H, W, C)
        pred_mask = preds_list[i]  # (H, W)
        
        # Prepare image for display
        img_vis, display_note = prepare_image_for_display(img, fix_rgb_order=False)
        cmap_image = "gray" if img_vis.ndim == 2 else None
        
        # Count detections
        unique_pred = np.unique(pred_mask)
        num_detections = len(unique_pred[unique_pred != 0])
        
        # Row 0: Original image
        axes[0, i].imshow(img_vis, cmap=cmap_image)
        axes[0, i].set_title(
            f"{filenames[i]}\n"
            f"Detections: {num_detections}\n"
            f"{display_note}",
            fontsize=11,
            fontweight="bold",
        )
        axes[0, i].axis("off")
        
        # Row 1: Inference with binary colormap (0=black, nonzero=red)
        pred_colored = create_binary_colormap(pred_mask)
        axes[1, i].imshow(pred_colored, vmin=0, vmax=1)
        axes[1, i].set_title(
            f"Inference (threshold={threshold})\n"
            f"{num_detections} instances detected",
            fontsize=11,
        )
        axes[1, i].axis("off")
    
    # Add figure title
    fig.suptitle(
        f"Inference Test on TIFF Images\n"
        f"Threshold: {threshold} | Total Detections: {sum(len(np.unique(p)[np.unique(p) != 0]) for p in preds_list)}",
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )
    
    plt.tight_layout()
    
    # Save figure
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()
    
    print(f"Saved visualization to: {output_path}")
    print(f"Total detections across all images: {sum(len(np.unique(p)[np.unique(p) != 0]) for p in preds_list)}")
    
    model.train()
    
    return images, preds_list

In [59]:
tiff_dir = "/explore/nobackup/people/rlgill/SystemTesting/lunar/newCubes/wac"

# Run visualization
images, predictions = visualize_inference_from_tiffs(
    model=model,
    image_processor=image_processor,
    device=device,
    tiff_dir=tiff_dir,
    mean=MEAN,
    std=STD,
    output_path="inference_test.png",
    n_images=20,
    threshold=0.5,
    normalize=False
)

Found 11 TIFF files
Loading 2 images from TIFF files...
Extracted item shape: (7, 512, 512)
Extracted item shape: (7, 512, 512)
(2, 7, 512, 512)


TypeError: cannot unpack non-iterable NoneType object

In [63]:
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
from glob import glob
from osgeo import gdal

tiffs = glob(f"{tiff_dir}/*.tif")
print(f"Found {len(tiffs)} TIFF files")

if len(tiffs) == 0:
    raise ValueError(f"No TIFF files found in {tiff_dir}")

n_samples = 1
n_bands = 100
tiffs = tiffs[:n_samples]

images = []
filenames = []

print(f"Loading {len(tiffs)} TIFF files...")
for tiff_path in tiffs:
    dataset = gdal.Open(tiff_path)
    num_bands = dataset.RasterCount

    print(f"Opened tiff with {num_bands} bands")
    
    # Check nodata values across bands
    print("\n" + "="*60)
    print("NODATA VALUE CHECK")
    print("="*60)
    
    nodata_values = []
    for band_num in range(1, min(n_bands + 1, num_bands + 1)):
        band = dataset.GetRasterBand(band_num)
        nodata = band.GetNoDataValue()
        nodata_values.append(nodata)
    
    unique_nodata = set(nodata_values)
    
    if len(unique_nodata) == 1:
        print(f"✓ All bands have the same nodata value: {nodata_values[0]}")
    else:
        print(f"⚠ Different nodata values found: {unique_nodata}")
        for i, nd in enumerate(nodata_values):
            if i == 0 or nd != nodata_values[i-1]:
                print(f"  Band {i+1}: {nd}")
    
    print("="*60 + "\n")
    
    # Read first n_bands (1-indexed in GDAL)
    band_list = list(range(1, n_bands+1))
    img_array = dataset.ReadAsArray(band_list=band_list)

    # Track mask changes
    previous_mask = None
    current_group_start = 0
    mask_change_bands = []
    
    print("\n" + "="*60)
    print("MASK CHANGE DETECTION")
    print("="*60)
    
    for i in range(0, n_bands, 25):
        fig, axes = plt.subplots(5, 5, figsize=(10, 10))
        bands_subset = img_array[i:i+25]
        band_idx = 0
        
        for row in range(5):
            for col in range(5):
                current_band_num = band_idx + i
                
                if band_idx < len(bands_subset) and current_band_num < n_bands:
                    band_to_plot = bands_subset[band_idx]
                    band_min, band_max = np.min(band_to_plot), np.max(band_to_plot)
                    print("Band min, max:", band_min, band_max)
                    
                    # Use the actual nodata value if available
                    nodata_val = nodata_values[current_band_num]
                    if nodata_val is not None:
                        mask = (band_to_plot == nodata_val) | np.isnan(band_to_plot) | np.isinf(band_to_plot)
                    else:
                        print("Nodata value is none, using (neg | nan | inf) mask.")
                        mask = (band_to_plot < 0) | np.isnan(band_to_plot) | np.isinf(band_to_plot)
                    
                    masked_band = ma.masked_where(mask, band_to_plot)
                    
                    if previous_mask is not None:
                        mask_diff = np.sum(mask != previous_mask)
                        total_pixels = mask.size
                        diff_percentage = (mask_diff / total_pixels) * 100
                        
                        if diff_percentage > 0.1:
                            bands_in_group = current_band_num - current_group_start
                            print(f"\n🔄 MASK CHANGE detected at Band {current_band_num + 1}")
                            print(f"   Previous group: Bands {current_group_start + 1}-{current_band_num} ({bands_in_group} bands)")
                            print(f"   Valid pixels before: {np.sum(~previous_mask):,} / {total_pixels:,} ({np.sum(~previous_mask)/total_pixels*100:.2f}%)")
                            print(f"   Valid pixels now: {np.sum(~mask):,} / {total_pixels:,} ({np.sum(~mask)/total_pixels*100:.2f}%)")
                            print(f"   Pixels changed: {mask_diff:,} ({diff_percentage:.2f}%)")
                            mask_change_bands.append(current_band_num)
                            current_group_start = current_band_num
                    
                    previous_mask = mask.copy()
                    
                    if masked_band.compressed().size > 0:
                        valid_data = masked_band.compressed()
                        vmin = np.percentile(valid_data, 2)
                        vmax = np.percentile(valid_data, 98)
                        
                        if vmin == vmax:
                            vmin = valid_data.min()
                            vmax = valid_data.max()
                            if vmin == vmax:
                                vmax = vmin + 1
                    else:
                        vmin, vmax = 0, 1
                    
                    cmap = plt.cm.viridis.copy()
                    cmap.set_bad(color='lightgray')
                    
                    im = axes[row, col].imshow(masked_band, cmap=cmap, vmin=vmin, vmax=vmax)
                    axes[row, col].set_title(f"Band {current_band_num + 1}", fontsize=8)
                    axes[row, col].axis("off")
                    fig.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)
                else:
                    axes[row, col].axis("off")
                
                band_idx += 1
        
        plt.tight_layout()
        plt.savefig(f"test_viz_bands_{i}_{i+25}.png", dpi=150, bbox_inches='tight')
        plt.close(fig)
    
    bands_in_final_group = n_bands - current_group_start
    print(f"\n   Final group: Bands {current_group_start + 1}-{n_bands} ({bands_in_final_group} bands)")
    
    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(f"Total bands: {n_bands}")
    print(f"Mask changes detected: {len(mask_change_bands)}")
    if len(mask_change_bands) > 0:
        print(f"Estimated number of stacked images: {len(mask_change_bands) + 1}")
        print(f"Mask change points at bands: {[b+1 for b in mask_change_bands]}")
    else:
        print("No mask changes detected - all bands appear to be from the same image")
    print("="*60 + "\n")

Found 11 TIFF files
Loading 1 TIFF files...
Opened tiff with 5229 bands

NODATA VALUE CHECK
✓ All bands have the same nodata value: None

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]

MASK CHANGE DETECTION
Band min, max: -3.4028227e+38 0.039273605
Nodata value is none, using (neg | nan | inf) mask.
Band min, max: -3.4028227e+38 0.04281393
Nodata value is none, using (neg | nan | inf) mask.

🔄 MASK CHANGE detected at Band 2
   Previous group: Bands 1-1 (1 bands)
   Valid pixels before: 10,376 / 262,144 (3.96%)
   Valid pixels now: 8,759 / 262,144 (3.34%)
   Pixels changed: 1,617 (0.62%)
Band min, max: -3.4028227e+38 0.062010452
Nodata value is n